In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

#### Functions

In [ ]:
class MinMaxScaler:
    def __init__(self):
        pass
    def fit(self, X):
        dict_fit = {}
        for col in X.columns:
            # get min
            flt_min = X[col].min()
            # get max
            flt_max = X[col].max()
            # get range
            flt_range = flt_max - flt_min
            # mak dict
            dict_tmp = {
                'min': flt_min,
                'range': flt_range,
            }
            # assign
            dict_fit[col] = dict_tmp
        # save to object
        self.dict_fit = dict_fit
        # return object
        return self
    def transform(self, X):
        for col, dict_tmp in self.dict_fit.items():
            flt_min = dict_tmp['min']
            flt_range = dict_tmp['range']
            X[f'{col}_scaled'] = (X[col] - flt_min) / flt_range
        # return X 
        return X

#### Constants

In [ ]:
str_dirname_output = './output'

#### Make output dir

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

#### Import data

In [ ]:
str_uri = 'uri_from_aws'
df = pd.read_parquet(str_uri)
df

#### List of features

In [ ]:
list_cols_model = [
    # put features here
]

#### Scale features

In [ ]:
X = df[list_cols_model].copy()
cls_scaler = MinMaxScaler()
# fit
cls_scaler.fit(X)
# transform
X = cls_scaler.transform(X)
# show
X

#### Get scaled features

In [ ]:
list_cols_scaled = [col for col in X.columns if '_scaled' in col]

#### Cluster analysis

In [ ]:
list_dict_row = []
for int_n_clusters in tqdm(range(2, 21)):
    # initialize
    cls_model_inference = KMeans(
        random_state=42,
        n_clusters=int_n_clusters,
    )

    # fit
    cls_model_inference.fit(
        X[list_cols_scaled].copy(),
    )
    
    # get inertia
    flt_inertia = cls_model_inference.inertia_
    
    # row
    dict_row = {
        'n_clusters': int_n_clusters,
        'inertia': flt_inertia,
    }
    # append
    list_dict_row.append(dict_row)

# df
df_tmp = pd.DataFrame(list_dict_row)

# show
df_tmp

#### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Inertia by Clusters')
ax.set_xlabel('N Clusters')
ax.set_ylabel('Inertia')
ax.plot(df_tmp['n_clusters'].astype(str), df_tmp['inertia'])
plt.show()

#### Optimal n clusters is elbow of plot

In [ ]:
# elbow
int_n_clusters = 7

# initialize
cls_model_inference = KMeans(
    random_state=42,
    n_clusters=int_n_clusters,
)

# fit
cls_model_inference.fit(
    df[list_cols_model].copy(),
)

# get inertia
flt_inertia = cls_model_inference.inertia_
print(f'Inertia: {flt_inertia:0.4f}')

# make label
df['cluster'] = cls_model_inference.labels_

# show
df

#### Cluster frequency

In [ ]:
ser_freq = df['cluster'].value_counts(normalize=True)
df_freq = ser_freq.reset_index()
df_freq.columns = ['cluster', 'proportion']
# sort
df_freq.sort_values(by='cluster', ascending=True, inplace=True)
# show
df_freq

#### Plot frequency

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.set_title('Proportion by Cluster')
ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion')
bars = ax.bar(df_freq['cluster'].astype(str), df_freq['proportion'])
ax.bar_label(bars, fmt='%0.2f')
plt.show()

#### Means by cluster

In [ ]:
dict_agg = {col: 'mean' for col in list_cols_model}
dict_agg['60dpd730'] = 'mean'
df_pivot = df.groupby(by='cluster', as_index=False).agg(dict_agg)
# show
df_pivot

#### Plot means

In [ ]:
list_cols = list(dict_agg.keys())
for col in list_cols:
    fig, ax = plt.subplots(figsize=(9,5))
    ax.set_title(f'Mean {col} by Cluster')
    ax.set_xlabel('Cluster')
    ax.set_ylabel(f'Mean {col}')
    bars = ax.bar(df_pivot['cluster'].astype(str), df_pivot[col])
    ax.bar_label(bars, fmt='%0.4f')
    # show
    plt.show()